# How to use Demographic Parity Loss to debias your ML model

In [1]:
import torch
from FairReg.Regularization.EqualizedOddsLoss import EqualizedOddsLoss
from Models.logistic_regression_net import LinearClassificationNet
import random
import numpy as np
import os
from utils.model_utils import ModelUtils
from FairReg.DPLUtils.regularization_config import RegularizationConfig
from FairReg.Learning.learning_new import Learning
from utils.dataset_utils import DatasetUtils

In [2]:
batch_size = 64

In [3]:
train_ds = torch.load("./train_ds")
test_ds = torch.load("./test_ds")

Using ['sex_binary'] as sensitive feature(s).


In [4]:
# # Compute the disparity of the training dataset
# max_disparity = 0
# for target in range(0,1):
#     for sensitive_feature in range(0,1):
#         max_disparity = max(RegularizationLoss().compute_violation_with_argmax(
#             predictions_argmax=train_ds.targets,
#             sensitive_attribute_list=train_ds.sensitive_features,
#             current_target=target,
#             current_sensitive_feature=sensitive_feature
#         ), max_disparity)
# print("Disparity of the training dataset: ", max_disparity)

In [5]:
train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

# Train a simple model without any Fairness Mitigation

In [1]:
# Create the model that we will train, for Dutch we will use a LinearClassificationNet
# defined inside this Library in the Models/logistic_regression_net.py file
model = LinearClassificationNet()
batch_size = 333
lr = 0.019925917176300392
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
epochs = 5
seed = 42

# seed the model
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
os.environ["PYTHONHASHSEED"] = str(seed)

NameError: name 'LinearClassificationNet' is not defined

In [7]:
# We don't want to use privacy in this case but we make the model private using
# the noise=0 because the code for learning the model only accept private models.
private_model, private_optimizer, private_train_loader = ModelUtils.create_private_model(
    model=model,
    epsilon=None,
    noise_multiplier=0,
    original_optimizer=optimizer,
    train_loader=train_loader,
    epochs=epochs,
    delta=0,
    MAX_GRAD_NORM=10000000000,  # since we just need to wrap the model without using privacy we use a high value here
    batch_size=batch_size,
)

/home/l.corbucci/Unfairness-Regularization/.venv/lib/python3.10/site-packages/opacus/privacy_engine.py:142: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


In [8]:
train_parameters = RegularizationConfig(
    epochs=epochs, device="cuda", batch_size=batch_size, seed=seed, optimizer="adam", regularization=False
)

In [9]:
model.to(train_parameters.device)

LinearClassificationNet(
  (layer1): Linear(in_features=11, out_features=2, bias=False)
)

In [10]:
for epoch in range(0, epochs):
    # Now we can train the model. First of all we will train a model without any
    # fairness mitigation
    results = Learning.train_private_model(
        train_parameters=train_parameters,
        model=private_model,
        model_regularization=None,
        optimizer=private_optimizer,
        optimizer_regularization=None,
        train_loader=train_loader,
        test_loader=test_loader,
        average_probabilities=None,
        current_epoch=epoch,
    )
    print(
        f"Epoch {epoch} - Train accuracy {results['Train Accuracy']} - Train Loss {results['Train Loss']} - Max Disparity Train {results['Max Disparity Train']}"
    )

/home/l.corbucci/Unfairness-Regularization/.venv/lib/python3.10/site-packages/torch/nn/modules/module.py:1359: UserWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  warnings.warn("Using a non-full backward hook when the forward contains multiple autograd Nodes "
/home/l.corbucci/Unfairness-Regularization/.venv/lib/python3.10/site-packages/numpy/core/fromnumeric.py:3464: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/l.corbucci/Unfairness-Regularization/.venv/lib/python3.10/site-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/l.corbucci/Unfairness-Regularization/DPL/Regularization/EqualizedOddsLoss.py:81: UserWarning: To copy construct from a te

[tensor(0.0183, device='cuda:0'), tensor(0.0183, device='cuda:0'), tensor(0.0183, device='cuda:0'), tensor(0.0183, device='cuda:0'), tensor(0.0190, device='cuda:0'), tensor(0.0190, device='cuda:0'), tensor(0.0190, device='cuda:0'), tensor(0.0190, device='cuda:0'), tensor(0.0381, device='cuda:0'), tensor(0.0381, device='cuda:0'), tensor(0.0381, device='cuda:0'), tensor(0.0381, device='cuda:0'), tensor(0.0426, device='cuda:0'), tensor(0.0426, device='cuda:0'), tensor(0.0426, device='cuda:0'), tensor(0.0426, device='cuda:0')]
Epoch 0 - Train accuracy 0.7969008684158325 - Train Loss 0.47623088368505395 - Max Disparity Train 0.04261159896850586
[tensor(0.0381, device='cuda:0'), tensor(0.0381, device='cuda:0'), tensor(0.0381, device='cuda:0'), tensor(0.0381, device='cuda:0'), tensor(0.0379, device='cuda:0'), tensor(0.0379, device='cuda:0'), tensor(0.0379, device='cuda:0'), tensor(0.0379, device='cuda:0'), tensor(0.0274, device='cuda:0'), tensor(0.0274, device='cuda:0'), tensor(0.0274, device

In [11]:
# We can test the trained model on the test dataset to understand if the model is less unfair than before
_, accuracy, _, _, _, max_disparity_test = Learning.test(
    model=private_model,
    test_loader=test_loader,
    train_parameters=train_parameters,
    current_epoch=epochs,
)

print(f"Test accuracy {accuracy} - Max Disparity Test {max_disparity_test}")

Test accuracy 0.8043694141012909 - Max Disparity Test 0.042308270931243896


# Train a model with Fairness Mitigation

In [12]:
# Create the model that we will train, for Dutch we will use a LinearClassificationNet
# defined inside this Library in the Models/logistic_regression_net.py file
model = LinearClassificationNet()
model_regularization = LinearClassificationNet()
batch_size = 123
lr = 0.03722692381416153
optimizer = torch.optim.SGD(model.parameters(), lr=lr)
optimizer_regularization = torch.optim.SGD(model_regularization.parameters(), lr=lr)
epochs = 5
seed = 42
# seed the model
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
os.environ["PYTHONHASHSEED"] = str(seed)

In [13]:
train_parameters = RegularizationConfig(
    epochs=epochs,
    device="cuda",
    batch_size=batch_size,
    seed=seed,
    regularization=True,
    target=0.05,
    regularization_mode="tunable",
    momentum=0.657142498624442,
    optimizer="adam",
    alpha=0.1,
)

In [14]:
# We don't want to use privacy in this case but we make the model private using
# the noise=0 because the code for learning the model only accept private models.

private_model, private_optimizer, private_train_loader = ModelUtils.create_private_model(
    model=model,
    epsilon=None,
    noise_multiplier=0,
    original_optimizer=optimizer,
    train_loader=train_loader,
    epochs=epochs,
    delta=0,
    MAX_GRAD_NORM=10000000000,  # since we just need to wrap the model without using privacy we use a high value here
    batch_size=batch_size,
)

/home/l.corbucci/Unfairness-Regularization/.venv/lib/python3.10/site-packages/opacus/privacy_engine.py:142: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


In [15]:
# We don't want to use privacy in this case but we make the model private using
# the noise=0 because the code for learning the model only accept private models.

private_model_regularization, private_optimizer_regularization, _ = ModelUtils.create_private_model(
    model=model_regularization,
    epsilon=None,
    noise_multiplier=0,
    original_optimizer=optimizer_regularization,
    train_loader=train_loader,
    epochs=epochs,
    delta=0,
    MAX_GRAD_NORM=10000000000,  # since we just need to wrap the model without using privacy we use a high value here
    batch_size=batch_size,
)

In [16]:
model.to(train_parameters.device)

LinearClassificationNet(
  (layer1): Linear(in_features=11, out_features=2, bias=False)
)

In [17]:
for epoch in range(0, epochs):
    # Now we can train the model. First of all we will train a model without any
    # fairness mitigation
    results = Learning.train_private_model(
        train_parameters=train_parameters,
        model=private_model,
        model_regularization=private_model_regularization,
        optimizer=private_optimizer,
        optimizer_regularization=private_optimizer_regularization,
        train_loader=private_train_loader,
        test_loader=test_loader,
        average_probabilities=None,
        current_epoch=epoch,
    )
    print(
        f"Epoch {epoch} - Train accuracy {results['Train Accuracy']} - Train Loss {results['Train Loss']} - Max Disparity Train {results['Max Disparity Train']}"
    )

/home/l.corbucci/Unfairness-Regularization/.venv/lib/python3.10/site-packages/torch/nn/modules/module.py:1359: UserWarning: Using a non-full backward hook when the forward contains multiple autograd Nodes is deprecated and will be removed in future versions. This hook will be missing some grad_input. Please use register_full_backward_hook to get the documented behavior.
  warnings.warn("Using a non-full backward hook when the forward contains multiple autograd Nodes "
/home/l.corbucci/Unfairness-Regularization/DPL/Regularization/EqualizedOddsLoss.py:81: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  predictions_argmax = torch.argmax(torch.tensor(predictions), dim=1).to(device)


Epoch 0 - Train accuracy 0.4832783639431 - Train Loss 0.02704091809656352 - Max Disparity Train 0.07416681945323944
Epoch 1 - Train accuracy 0.5559032559394836 - Train Loss 0.01659516172391269 - Max Disparity Train 0.003353993408381939
Epoch 2 - Train accuracy 0.5637576580047607 - Train Loss 0.03660667740247109 - Max Disparity Train 0.03499940037727356
Epoch 3 - Train accuracy 0.6600739359855652 - Train Loss 0.090310297379349 - Max Disparity Train 0.036362677812576294
Epoch 4 - Train accuracy 0.6884194612503052 - Train Loss 0.04983410721443862 - Max Disparity Train 0.00016802549362182617


In [18]:
# We can test the trained model on the test dataset to understand if the model is less unfair than before
_, accuracy_test_debias, _, _, _, max_disparity_test_debias = Learning.test(
    model=private_model,
    test_loader=test_loader,
    train_parameters=train_parameters,
    current_epoch=epochs,
)

print(f"Test accuracy {accuracy_test_debias} - Max Disparity Test {max_disparity_test_debias}")

Test accuracy 0.520191989407481 - Max Disparity Test 0.00014954805374145508


/home/l.corbucci/Unfairness-Regularization/.venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


# Comparison

| Unfairness Mitigation  | Accuracy  | Final Disparity  |
|---|---|---|
| False  | 0.8160 | 0.1586 |
| True | 0.7224 | 0.0165 |

Please note that the results may be improved with an hyperparameter search. For these two examples I've used the hyperapameters of a different experiment.